In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
!pip install MDAnalysis

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.9/108.9 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 37.0 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 56.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 2.8 MB/s eta 0:00:00


In [ ]:
# This code analyses the environment around CA (within 5A) with MDAnslysis,
# and creates tables with frequency of appearance of one atom within 5A throughout the simulation. 

# Created by: Anna Perova

#IMPORTS
import os
from pathlib import Path
import pandas as pd
import MDAnalysis as mda

#SETUP
rootdir = '/content/drive/MyDrive/M1_STAGE/Data/simulations_1HSI/'
os.chdir(rootdir)

#FUNCTIONS
def get_dynamic_environment(gro, xtc, cutoff=5.0):
    u = mda.Universe(gro, xtc)
    # Define your targets: ASN 40 on both chains
    # In MDA, we can use segids or chainIDs instead of atom IDs
    CA_asn_chB = u.select_atoms("name CA and resnum 40 and index 1536:3073")
    #0:1537
    #1536:3073

    # We want to find atoms around them
    env_atoms = set()

    #print(f"Analyzing {len(u.trajectory)} frames...")
    for ts in u.trajectory[:]:
        # Find atoms within 'cutoff' of the ASN
        around = u.select_atoms(f"around {cutoff} group target", target=CA_asn_chB)
        
        # Add the unique atom names/numbers found in this frame to our set
        for atm in around.atoms:
            env_atoms.add((atm.name, atm.resname, atm.resnum, atm.index))


    # Sort and print the results
    sorted_env = sorted(list(env_atoms), key=lambda x: x[1])
    return sorted_env

def analyze_env_frequency(gro, xtc, cutoff=5.0):
    u = mda.Universe(gro, xtc)
    CA_asn_chB = u.select_atoms("name CA and resnum 40 and index 1536:3073") # Adjusted for chain B
    
    # Dictionary to count how many frames each residue is present
    atoms_counts = {}
    n_frames = 0

    for ts in u.trajectory[::1]:
        n_frames += 1
        # Find residues within cutoff (excluding the Asn itself)
        neighbors = u.select_atoms(f"around {cutoff} group CA_asn_chB", CA_asn_chB=CA_asn_chB).residues - CA_asn_chB.residues   
        
        for atm in neighbors.atoms:
            atm_id = f"{atm.index}_{atm.name}_{atm.resname}_{atm.resnum}"
            atoms_counts[atm_id] = atoms_counts.get(atm_id, 0) + 1
        

    # Convert to percentage
    freq_data = [{"Residue": k, "Frequency": (v / n_frames) * 100} for k, v in atoms_counts.items()]
    return pd.DataFrame(freq_data).sort_values(by="Frequency", ascending=False)


# MAIN

# 0. Check
#env = get_dynamic_environment("/home/sdv/m1isdd/aperova/Documents/M1_STAGE/Data/simulations_1HSI/simulations_1HSI_APO_DP/V1/V1.gro", "/home/sdv/m1isdd/aperova/Documents/M1_STAGE/Data/simulations_1HSI/simulations_1HSI_APO_DP/V1/md_V1.xtc")
#freq_env = analyze_env_frequency("/home/sdv/m1isdd/aperova/Documents/M1_STAGE/Data/simulations_1HSI/simulations_1HSI_APO_DP/V1/V1.gro", "/home/sdv/m1isdd/aperova/Documents/M1_STAGE/Data/simulations_1HSI/simulations_1HSI_APO_DP/V1/md_V1.xtc")
#print("Interesting atoms within 5A:", pd.DataFrame(env))
#print("Frequency of atoms appearing within 5A:", pd.DataFrame(freq_env))



# 1. Fetch files from all simulations
sim_files = [] # For all simulations

for subdir, dirs, files in os.walk(rootdir):
    gro_files_pathnames = [f for f in files if f.endswith(".gro")]
    xtc_files_pathnames = [f for f in files if f.endswith(".xtc")]
    for gro_name in gro_files_pathnames:
        sim_name_match = gro_name.replace(".gro", "")
        found_xtc = None
        for xtc_name in xtc_files_pathnames:
            if sim_name_match in xtc_name  and xtc_name.endswith('.xtc'):
                found_xtc = xtc_name
                break
        if found_xtc:
            gro_path = os.path.join(subdir, gro_name)
            xtc_path = os.path.join(subdir, found_xtc)
            sim_files.append((gro_path, xtc_path))

# 1.1. sort file pairs
sim_files.sort()
all_sim_dist = []
for gro, xtc in sim_files:
    sim_name = os.path.basename(os.path.dirname(gro))
    Asn_env_df = pd.DataFrame(get_dynamic_environment(gro, xtc), columns=['atomname', 'resname', 'resid', 'atomic_index'])
    Asn_env_df = Asn_env_df.sort_values(by=['atomic_index']).reset_index(drop=True)

    # 2.7. Write csv with ASN env residues
    src_path = r"/content/drive/MyDrive/M1_STAGE/Manips/"
    p = Path(src_path).parent.joinpath(f"Manips/Tables/{sim_name}_40_chB_env_atoms.csv")
    Asn_env_df.to_csv(p, index=True, header=True, decimal=".", float_format="  %.3f")
    # 2.8. Calculate frequencies
    Asn_env_freq_df = analyze_env_frequency(gro, xtc)
    p = Path(src_path).parent.joinpath(f"Manips/Tables/{sim_name}_40_chB_env_atoms_frequency.csv")
    Asn_env_freq_df.to_csv(p, index=True, header=True, decimal=".", float_format="  %.3f")

/usr/local/lib/python3.12/dist-packages/MDAnalysis/coordinates/XDR.py:261: UserWarning: Reload offsets from trajectory
 ctime or size or n_atoms did not match
  warnings.warn(


In [7]:
#SETUP
rootdir = '/content/drive/MyDrive/M1_STAGE/Manips/Tables/'
os.chdir(rootdir)


#MAIN
# 1. Extract frequency data

# 1. Read data

# 1.0. Setup
df_list = []
CA_atoms = []
simulation_names = ['V1', 'V11', 'V12', 'V7', 'V8', 'V21']
all_results = {}
residue_dico = {}
# Initialize a set to collect all unique atom identifiers across all simulations
all_unique_atom_identifiers = set()

# 1.1. Open frequency tables, read as df
freq_df_paths = [
    "V1_ASN_env_atoms_frequency.csv",
    "V11_ASN_env_atoms_frequency.csv",
    "V12_ASN_env_atoms_frequency.csv",
    "V7_ASN_env_atoms_frequency.csv",
    "V8_ASN_env_atoms_frequency.csv",
    "V21_ASN_env_atoms_frequency.csv"
]
for sim in freq_df_paths:
    sim_name = sim.replace("_ASN_env_atoms_frequency.csv", "")
    df_sim = pd.read_csv(sim)

    # 1.2. Filter only alpha carbons (and that are not Asp25 alpha carbon)
    filtered_CA = df_sim[
        df_sim["Residue"].apply(lambda x: 'CA' in x and x.endswith('CA_ASN_40') is False)
        ].copy()

    # 1.3. Save to new df for graph parameters
    df_new = pd.DataFrame()
    df_new["Atom_Identifier"] = filtered_CA["Residue"] # Keep the full atom identifier
    df_new["Frequency"] = filtered_CA['Frequency']

    # 1.3.1. For each atom, get its residue_id, chain, and atom_name
    chain_label_list = []
    res_id_list = []
    atom_name_list = []


    for atom in filtered_CA["Residue"].values:
        res_str = atom.split('_')
        atom_id = int(res_str[0])
        res_id = '_'.join(res_str[2:])
        atom_name = res_str[1]
        if atom_id < 1536:
            chain_label_list.append("Chain A")
        else:
            chain_label_list.append("Chain B")
        res_id_list.append(res_id)
        # Collect all unique atom identifiers in the global set
        atom_name_list.append(atom_name)
        all_unique_atom_identifiers.add(atom) # Add to the global set

    df_new["Chain"] = chain_label_list
    df_new["Residue_Id"] = res_id_list
    df_new["Atom"] = atom_name_list


    all_results[sim_name] = df_new



# Prepare a list to store parsed atom data
all_atom_data = []

# Iterate through all_results to gather and parse atom identifiers
for sim_name, df_sim_data in all_results.items():
    for _, row in df_sim_data.iterrows():
        atom_identifier = row['Atom_Identifier']
        parts = atom_identifier.split('_')
        atom_id = int(parts[0])
        atom_name = parts[1]
        residue_type = parts[2]
        residue_num = parts[3]

        chain_label = 'chA' if atom_id < 1536 else 'chB'

        # Create a 'descriptive_label' that excludes the atom_id
        descriptive_label = f"{atom_name}_{residue_type}{residue_num}_{chain_label}"

        all_atom_data.append({
            'Atom_Identifier_Full': atom_identifier,
            'Atom_ID': atom_id,
            'Descriptive_Label': descriptive_label,
            'Simulation': sim_name
        })

# Create a DataFrame from the collected data
atom_data_df = pd.DataFrame(all_atom_data)

# Group by Descriptive_Label and check for multiple unique Atom_IDs
duplicate_label_groups = atom_data_df.groupby('Descriptive_Label').filter(lambda x: x['Atom_ID'].nunique() > 1)

# Sort the results for better readability
duplicate_label_groups = duplicate_label_groups.sort_values(by=['Descriptive_Label', 'Atom_ID'])

#print("Atoms with the same descriptive label but different Atom_IDs (potential shifts):")
#display(duplicate_label_groups[['Descriptive_Label', 'Atom_ID', 'Atom_Identifier_Full', 'Simulation']])


# The 'atom_data_df' created in cell 'e0f79337' contains the 'Descriptive_Label' column
# Define the list of unique descriptive labels for the heatmap columns
all_residues = atom_data_df['Descriptive_Label'].unique().tolist()

# Initialize the heatmap_data DataFrame with simulation names as index and merged descriptive labels as columns
heatmap_data = pd.DataFrame(index=simulation_names, columns=all_residues)

# Populate the heatmap_data by aggregating frequencies for each simulation
for sim_name in simulation_names:
    # Get the raw frequency data for the current simulation
    df_sim_raw = all_results[sim_name].copy()

    # Add the Descriptive_Label column to this simulation's DataFrame if it doesn't exist
    # This ensures consistency with how 'all_residues' was generated
    descriptive_labels_for_sim = []
    for atom_identifier in df_sim_raw['Atom_Identifier']:
        parts = atom_identifier.split('_')
        atom_id_val = int(parts[0])
        atom_name = parts[1]
        residue_type = parts[2]
        residue_num = parts[3]
        chain_label = 'chA' if atom_id_val < 1536 else 'chB'
        descriptive_labels_for_sim.append(f"{atom_name}_{residue_type}{residue_num}_{chain_label}")
    df_sim_raw['Descriptive_Label'] = descriptive_labels_for_sim

    # Group by the Descriptive_Label and take the maximum frequency
    merged_frequencies = df_sim_raw.groupby('Descriptive_Label')['Frequency'].max()

    # Assign these merged frequencies to the correct row in heatmap_data
    for label, freq in merged_frequencies.items():
        if label in heatmap_data.columns: # Ensure the label exists in our defined columns
            heatmap_data.loc[sim_name, label] = freq

# Convert to float to handle NaN values properly
heatmap_data = heatmap_data.astype(float)
heatmap_data = heatmap_data.fillna(0)

# Sort the columns (residues) alphabetically for consistent display
#heatmap_data = heatmap_data.reindex(columns=sorted(heatmap_data.columns))


fig, ax = plt.subplots(figsize=(20, 15)) # Significantly increased figure height for larger squares and better visibility

sns.heatmap(heatmap_data.T, # Transpose the DataFrame to swap axes
            annot=False,        # Keep as False for now to avoid clutter with many small cells
            fmt=".1f",          # Format for annotations
            cmap="Blues",      # Color map (green for high frequency, red for low)
            square=True,       # Set to True if you want squares to be equal size
            cbar=True,  # Color of the lines between cells
            ax=ax,              # Pass the ax object to sns.heatmap
            cbar_kws={'label': 'Frequency (%)'}
            )

plt.title("Frequency of Interactions with Asn40 (chain B)", fontsize=16, y=1.05, x=0.66)
plt.xlabel("Simulation", fontsize=12) # Swapped x-axis label
plt.ylabel("Residue (within 5A)", fontsize=12) # Swapped y-axis label

# Use the 'all_residues' (which now contains the unique descriptive labels) directly for y-axis labels
formatted_y_labels = heatmap_data.columns.tolist() # Get the sorted columns from heatmap_data

# Remove 'CA_' prefix from the labels
formatted_y_labels = [label.replace('CA_', '') for label in formatted_y_labels]

plt.yticks(ticks=[x + 0.5 for x in range(len(formatted_y_labels))], labels=formatted_y_labels, rotation=0, fontsize=12) # Apply to y-axis with appropriate rotation, slightly increased fontsize
plt.xticks(ticks=[x + 0.5 for x in range(len(simulation_names))], labels=simulation_names, rotation=45, fontsize=15) # Apply to x-axis, using simulation_names
#plt.tight_layout() # Adjust layout to prevent labels from overlapping
plt.show()
fig.savefig("/content/drive/MyDrive/M1_STAGE/Manips/Figures/env_40_chB_heatmap.png", bbox_inches='tight', dpi=300)


FileNotFoundError: [Errno 2] No such file or directory: 'V1_ASN_env_atoms_frequency.csv'